In [ ]:
!pip -q install vaderSentiment

In [ ]:
import re
import nltk
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
for p in ["punkt", "punkt_tab", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    nltk.download(p, quiet=True)
vader = SentimentIntensityAnalyzer()

In [ ]:
reviews = [
 "The battery life is excellent and lasts two days.",
 "Camera quality is amazing, photos are sharp and bright.",
 "The screen is gorgeous but the price is too high.",
 "Battery drains very fast, really disappointed.",
 "Performance is smooth and the processor is very fast.",
 "Delivery was late and the packaging was damaged.",
 "Great value for money, the price is reasonable.",
 "The camera is terrible in low light.",
 "Customer support was helpful and solved my problem quickly.",
 "Screen cracked within a week, poor build quality.",
 "Love the display and the sound quality is superb.",
 "The phone heats up and the performance is laggy.",
 "Excellent battery backup and fast charging.",
 "Overpriced for the features it offers.",
 "Delivery was quick and the seller was polite.",
 "The camera is good but the battery is average.",
 "Worst purchase ever, the device stopped working.",
 "Sound quality is decent and the design looks premium.",
 "Fast performance, smooth display, totally worth the price.",
 "Poor customer support, nobody answered my calls.",
]
df = pd.DataFrame({"review": reviews})
# To use your own data:  df = pd.read_csv("reviews.csv")  # needs a column named 'review'
print(len(df), "reviews")

In [ ]:
df["score"] = df.review.apply(lambda t: vader.polarity_scores(t)["compound"])
df["sentiment"] = pd.cut(df.score, [-1.01, -0.05, 0.05, 1.01], labels=["Negative", "Neutral", "Positive"])
display(df.head(8))
counts = df.sentiment.value_counts()
counts.plot(kind="bar", color=["seagreen", "indianred", "gray"][:len(counts)], title="Overall sentiment")
plt.show()

In [ ]:
stop_nouns = {"phone", "device", "week", "days", "problem", "purchase", "features", "nobody", "calls", "seller", "photos", "light"}
nouns = []
for r in df.review:
    for w, tag in nltk.pos_tag(re.findall(r"[a-zA-Z]+", r.lower())):
        if tag.startswith("NN") and w not in stop_nouns and len(w) > 3:
            nouns.append(w)
print("Most frequent candidate aspects:", Counter(nouns).most_common(10))

In [ ]:
aspects = {
    "battery": ["battery", "charging", "backup"],
    "camera": ["camera", "photos"],
    "screen/display": ["screen", "display"],
    "price": ["price", "value", "overpriced"],
    "performance": ["performance", "processor", "laggy", "heats"],
    "delivery": ["delivery", "packaging", "seller"],
    "support": ["support", "customer"],
    "sound": ["sound"],
}
rows = []
for asp, keys in aspects.items():
    scs = [vader.polarity_scores(r)["compound"] for r in df.review if any(k in r.lower() for k in keys)]
    if scs:
        rows.append((asp, len(scs), sum(scs) / len(scs)))
asp_df = pd.DataFrame(rows, columns=["aspect", "mentions", "avg_sentiment"]).sort_values("avg_sentiment")
display(asp_df.round(3))

plt.figure(figsize=(7, 4))
plt.barh(asp_df.aspect, asp_df.avg_sentiment, color=["indianred" if v < 0 else "seagreen" for v in asp_df.avg_sentiment])
plt.axvline(0, color="k", lw=0.8); plt.title("Average sentiment per aspect"); plt.show()

In [ ]:
lex = vader.lexicon
pos_w, neg_w = Counter(), Counter()
for r in df.review:
    for w in re.findall(r"[a-z]+", r.lower()):
        if lex.get(w, 0) >= 1.5: pos_w[w] += 1
        elif lex.get(w, 0) <= -1.5: neg_w[w] += 1
print("Top positive opinion words:", pos_w.most_common(8))
print("Top negative opinion words:", neg_w.most_common(8))

best, worst = df.loc[df.score.idxmax()], df.loc[df.score.idxmin()]
print("\nMost positive review:", best.review, round(best.score, 2))
print("Most negative review:", worst.review, round(worst.score, 2))